# `ingestion/index.py` — Validation Walkthrough

Validates the chunking, embedding, and ChromaDB indexing pipeline against the FE 501s manual.
Covers chunk structure, edge cases fixed during development, and retrieval quality.

In [1]:
import sys, json
sys.path.insert(0, "..")

from dotenv import load_dotenv
load_dotenv()

from collections import Counter
from ingestion.index import build_chunks, parse_manual, parse_toc
import pypdf, chromadb, re
from openai import OpenAI

PDF        = "../manuals/FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du.pdf"
DB_PATH    = "../data/chroma"
EMBED_MODEL = "text-embedding-3-small"

pages  = parse_manual(PDF, extract_images=False)
toc    = parse_toc(pypdf.PdfReader(PDF))
chunks = build_chunks(pages, toc, "FE_501s_2026")

print(f"Pages:   {len(pages)}")
print(f"TOC entries: {len(toc)}")
print(f"Chunks:  {len(chunks)}")

Pages:   378
TOC entries: 254
Chunks:  600


## 1. Chunk distribution by phase

Verify the three-level hierarchy is producing the expected split across phase types.

In [2]:
dist = Counter(c.phase or "(none)" for c in chunks)
print(f"{'Phase':<15}  {'Chunks':>7}  {'%':>6}")
print("-" * 32)
for phase in ["(none)", "preparatory", "main", "reworking"]:
    n = dist[phase]
    print(f"{phase:<15}  {n:>7}  {100*n/len(chunks):>5.1f}%")
print("-" * 32)
print(f"{'Total':<15}  {len(chunks):>7}")

Phase             Chunks       %
--------------------------------
(none)               328   54.7%
preparatory           85   14.2%
main                 111   18.5%
reworking             76   12.7%
--------------------------------
Total                600


## 2. Chunk structure spot-check

Inspect a complete chunk — all metadata fields — for a known section with torque specs and figure refs.

In [3]:
# Section 7.2 — Adjusting the handlebar position
# Validates both bug fixes: 7.1 content must NOT appear, and M8 20 Nm spec must appear
sample = next(c for c in chunks if c.section == "7.2" and c.chapter_num == 7)

print(f"id            : {sample.id}")
print(f"manual        : {sample.manual}")
print(f"chapter       : {sample.chapter_num} — {sample.chapter_title}")
print(f"section       : {sample.section} — {sample.section_title}")
print(f"phase         : {sample.phase or '(none)'}")
print(f"pages         : {sample.page_nums}")
print(f"figure refs   : {sample.figure_refs}")
print(f"image paths   : {len(sample.image_paths)} file(s)  (populated by index_manual, empty here)")
print(f"torque specs  :")
for t in sample.torque_specs:
    note = f"  [{t['note']}]" if t.get("note") else ""
    print(f"  {t['bolt']:<8} {t['nm']} Nm  ({t['ftlbf']} ft·lbf)  {t['description']}{note}")

print(f"\n7.1 content in 7.2 chunk: {'7.1 Handlebar position' in sample.text}")
print(f"M8 handlebar clamp spec present: {any(t['bolt'] == 'M8' and 'clamp' in t['description'].lower() for t in sample.torque_specs)}")
print(f"\ntext preview  :\n{sample.text[:300]}")

id            : FE_501s_2026__7__7.2__full
manual        : FE_501s_2026
chapter       : 7 — Handlebar, controls
section       : 7.2 — Adjusting the handlebar position
phase         : (none)
pages         : [50, 51]
figure refs   : ['W00323-10']
image paths   : 0 file(s)  (populated by index_manual, empty here)
torque specs  :
  M10      40.0 Nm  (29.5 ft·lbf)  Screw, handlebar mount  [Loctite® 243]
  M8       20.0 Nm  (14.8 ft·lbf)  Handlebar clamp screw  [Make sure the installed gap widths are even.]

7.1 content in 7.2 chunk: False
M8 handlebar clamp spec present: True

text preview  :
7.2 Adjusting the handlebar position
WARNING
Danger of accidents A repaired handlebar poses a safety risk.
If the handlebar is bent or straightened, the material becomes fatigued. The handlebar may break as a
result.
‒ Change the handlebar if the handlebar is damaged or bent.
W00323-10
‒ Remove (1) 


## 3. Edge cases fixed during development

Six bugs were caught and fixed while building the index. This section reproduces the conditions that exposed each one.

### 3a. Duplicate phase IDs in multi-page sections

Some sections span multiple pages, causing the same phase header to appear more than once in the
combined text. The chunker appends a `_0`, `_1` suffix when a phase repeats within a section.

In [4]:
suffixed = [c for c in chunks if re.search(r"__(preparatory|main|reworking|full)_\d+$", c.id)]
print(f"Chunks with repeated-phase suffix: {len(suffixed)}")
for c in suffixed[:6]:
    print(f"  {c.id}")

Chunks with repeated-phase suffix: 3
  FE_501s_2026__10__10.5__preparatory_1
  FE_501s_2026__10__10.5__main_1
  FE_501s_2026__10__10.5__reworking_1


### 3b. TOC pages leaking into section buckets

TOC pages (pp. 4–8) have parseable chapter headers and section numbers from the entries listed on
them. Without filtering, these collide with real content pages. Fix: skip pages containing
`"Table of contents"` before bucketing.

In [5]:
toc_pages = [p for p in pages if "Table of contents" in p.text and p.chapter_num is not None]
print(f"TOC pages that would have leaked (now filtered): {[p.page_num for p in toc_pages]}")
print(f"Sample section numbers on TOC page {toc_pages[0].page_num}: {toc_pages[0].section!r}")

TOC pages that would have leaked (now filtered): [4, 5, 6, 8]
Sample section numbers on TOC page 4: '9.3'


### 3c. Measurement values matching the section regex

Decimal measurements like `"1.0 mA"` or `"0.5 mm"` at the start of a line matched the original
section regex `^\d+\.\d+\s`. Fix: tighten to require a capital letter after the number
(`^\d+\.\d+\s+[A-Z]`), since all real section headings start with a capitalised title word.

In [6]:
import re

old_re = re.compile(r"^(\d+(?:\.\d+)+)\s",    re.MULTILINE)
new_re = re.compile(r"^(\d+(?:\.\d+)+)\s+[A-Z]", re.MULTILINE)

# Find lines that old regex matched but new one doesn't (false positives)
false_positives = set()
for p in pages:
    old_hits = set(old_re.findall(p.text))
    new_hits = set(new_re.findall(p.text))
    false_positives |= old_hits - new_hits

print(f"Section numbers only old regex matched (false positives): {sorted(false_positives)}")

Section numbers only old regex matched (false positives): ['0.00323', '0.00335', '0.0059', '0.0067', '0.0177', '0.0315', '0.082', '0.085', '0.20', '0.40', '0.6', '0.95', '1.0', '1.2', '1.4', '1.7', '1.8', '1.8953', '1.9346', '10.6', '15.525', '165.3', '17.6', '187.4', '209.4', '22.1', '27.1.2', '27.2.2', '3.7386', '3.7390', '3.74063', '3.74114', '40.7', '47.84', '48.14', '49.07', '49.14', '52.94', '54.91', '94.96', '94.97', '95.012', '95.025']


### 3d. Section content bleeding across page boundaries

Pages were originally assigned to their **last** section heading, so a page containing both 7.1 and
7.2 content was bucketed entirely under 7.2. Fix: each page is now split at section heading
boundaries before bucketing, so pre-heading text stays with the section that started on a previous
page.

In [7]:
c71 = next(c for c in chunks if c.section == "7.1" and c.chapter_num == 7)
c72 = next(c for c in chunks if c.section == "7.2" and c.chapter_num == 7)

print("Section 7.1 chunk starts with:")
print(c71.text[:120])
print()
print("Section 7.2 chunk starts with:")
print(c72.text[:120])
print()
print(f"7.1 content in 7.2 chunk: {'7.1 Handlebar position' in c72.text}  ← must be False")

Section 7.1 chunk starts with:
7.1 Handlebar position
W00322-10
The holes on the handlebar support are placed at a distance
of (A) from the center.
Hol

Section 7.2 chunk starts with:
7.2 Adjusting the handlebar position
WARNING
Danger of accidents A repaired handlebar poses a safety risk.
If the handle

7.1 content in 7.2 chunk: False  ← must be False


### 3e. Cross-page torque specs not captured

The `M8 20 Nm` handlebar clamp spec from section 7.2 appeared at the top of page 51, but page 51
was assigned to section 7.4 under the old bucketing. Fix: torque specs (and figure refs) are now
re-extracted from the combined section text after bucketing, not aggregated from individual pages.

In [8]:
c72 = next(c for c in chunks if c.section == "7.2" and c.chapter_num == 7)
clamp_spec = next((t for t in c72.torque_specs if "clamp" in t["description"].lower()), None)

print("Torque specs in section 7.2:")
for t in c72.torque_specs:
    print(f"  {t['bolt']:<8} {t['nm']} Nm  {t['description']}")
print()
print(f"Handlebar clamp spec captured: {clamp_spec is not None}  ← must be True")
if clamp_spec:
    print(f"  {clamp_spec['bolt']} {clamp_spec['nm']} Nm ({clamp_spec['ftlbf']} ft·lbf)  {clamp_spec['description']}")

Torque specs in section 7.2:
  M10      40.0 Nm  Screw, handlebar mount
  M8       20.0 Nm  Handlebar clamp screw

Handlebar clamp spec captured: True  ← must be True
  M8 20.0 Nm (14.8 ft·lbf)  Handlebar clamp screw


### 3f. Inline diagram reference markers concatenated with words

Diagram callout labels (e.g. `screws1`, `ofA`) were embedded directly in surrounding text with no
separator, making them unreadable. Fix: `_clean_text` in `parse.py` now expands these to
parenthetical form — `screws (1)`, `of (A)` — using a regex that matches letter sequences ending
in lowercase followed by digits or uppercase letters, leaving bolt specs (`M8`, `M10`) and
all-caps words (`WARNING`) untouched.

In [9]:
p50 = next(p for p in pages if p.page_num == 50)

# Spot-check the reported example and a few others
checks = [
    ("of (A)",           "ofA → of (A)"),
    ("distance (A)",     "distanceA → distance (A)"),
    ("Remove (1)",       "Remove1 → Remove (1)"),
    ("screws (2)",       "screws2 → screws (2)"),
    ("M10",              "M10 bolt spec unchanged"),
    ("WARNING",          "WARNING all-caps unchanged"),
]
for expected, label in checks:
    present = expected in p50.text
    status  = "✓" if present else "✗"
    print(f"{status}  {label}")

✓  ofA → of (A)
✓  distanceA → distance (A)
✓  Remove1 → Remove (1)
✓  screws2 → screws (2)
✓  M10 bolt spec unchanged
✓  WARNING all-caps unchanged


## 4. ChromaDB collection stats

Verify the collection count matches what was built, and inspect the stored metadata schema.

In [10]:
chroma = chromadb.PersistentClient(path=DB_PATH)
col    = chroma.get_collection("manuals")

print(f"Collection count: {col.count()}  (built {len(chunks)} chunks)")

# Peek at stored metadata fields for one record
peek = col.get(limit=1, include=["metadatas"])
meta = peek["metadatas"][0]
print(f"\nMetadata keys: {list(meta.keys())}")
print(f"\nSample record:")
for k, v in meta.items():
    display_v = v if len(str(v)) < 80 else str(v)[:77] + "..."
    print(f"  {k:<15}: {display_v}")

Collection count: 600  (built 600 chunks)

Metadata keys: ['section', 'chapter_num', 'image_paths', 'manual', 'chapter_title', 'phase', 'figure_refs', 'section_title', 'torque_specs', 'page_nums']

Sample record:
  section        : 1.1
  chapter_num    : 1
  image_paths    : []
  manual         : FE_501s_2026_US_en_Bundle_RM_069969-000001_05sq_m1du
  chapter_title  : Means of representation
  phase          : 
  figure_refs    : []
  section_title  : Conventions
  torque_specs   : []
  page_nums      : [9]


## 5. Retrieval quality

Run a few natural-language queries and check that the top results match the expected sections.

In [11]:
openai_client = OpenAI()

def query(text: str, n: int = 3) -> None:
    emb = openai_client.embeddings.create(model=EMBED_MODEL, input=[text]).data[0].embedding
    results = col.query(query_embeddings=[emb], n_results=n)
    print(f"Query: \"{text}\"")
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        phase = f" [{meta['phase']}]" if meta["phase"] else ""
        print(f"  ch{meta['chapter_num']} {meta['section']}{phase}  {meta['section_title']}")
    print()

query("remove the rear shock absorber")
query("valve clearance adjustment")
query("torque spec for swingarm pivot bolt")
query("change engine oil and oil filter")

Query: "remove the rear shock absorber"
  ch9 9.9  Removing the shock absorber
  ch14 14.8.1  Removing the rear wheel
  ch9 9.13  Spring, removing

Query: "valve clearance adjustment"
  ch24 24.1  Checking the valve clearance
  ch24 24.2  Adjusting the valve clearance
  ch18 18.5.6  Checking the valve clearance

Query: "torque spec for swingarm pivot bolt"
  ch27 2.6  
  ch27 27.6  Tightening torque
  ch27 2.7  

Query: "change engine oil and oil filter"
  ch18 18.3.3  
  ch22 22.3  
  ch22 22.5 [main]  Checking the oil pressure

